In [1]:
pip install sdv sdmetrics

  Using cached sdv-1.37.0-py3-none-any.whl.metadata (14 kB)
  Using cached sdmetrics-0.28.0-py3-none-any.whl.metadata (9.9 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached graphviz-0.21-py3-none-any.whl.metadata (12 kB)
  Using cached pandas-2.3.3-cp311-cp311-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (91 kB)
  Using cached copulas-0.14.1-py3-none-any.whl.metadata (9.7 kB)
  Using cached ctgan-0.12.1-py3-none-any.whl.metadata (10 kB)
  Using cached deepecho-0.8.1-py3-none-any.whl.metadata (11 kB)
  Using cached rdt-1.21.0-py3-none-any.whl.metadata (11 kB)
  Using cached jmespath-1.1.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached s3transfer-0.18.0-py3-none-any.whl.metadata (1.7 kB)
  Using cached pytz-2026.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2026.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached scikit_learn-1.9.0-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using c

# Libreria SDV 
*Autor: Carlos*
*Fecha: 04 junio*

In [1]:
import pandas as pd 
import numpy as np 
import random 

In [2]:
import sdv # Generar datos sintéticos
import sdmetrics # Evaluar la calidad de los datos
from sdv.metadata import SingleTableMetadata
from sdv.single_table import GaussianCopulaSynthesizer


In [3]:
from sdmetrics.reports.single_table import QualityReport

In [4]:
print("SDV", sdv.__version__)

SDV 1.37.0


In [5]:
# Creamos un dataset (original) que servira como fuente para crear los datos sintéticos
dfClient = pd.DataFrame({
    "cliente_id": [1,2,3,4,5,6,7,8,9,10],
    "edad": [23,33,43,28,53,56,43,56,65,40],
    "ingreso_mensual": [25000, 15000, 20000, 10000, 5000, 17000, 30000, 12000, 35000, 7500],
    "ciudad": ["Cordoba", "Veracruz", "Paso del Macho", "Amatlan", "Fortin", "Cuitlahuac", "Yanga", "Orizaba", "Cuitlahuac", "Potrero"]
})

In [6]:
dfClient.head()

,cliente_id,edad,ingreso_mensual,ciudad
0,1,23,25000,Cordoba
1,2,33,15000,Veracruz
2,3,43,20000,Paso del Macho
3,4,28,10000,Amatlan
4,5,53,5000,Fortin


In [8]:
# Definir la metadata
metadata = SingleTableMetadata()

In [9]:
metadata.detect_from_dataframe(
    data = dfClient
)

In [11]:
metadata.to_dict()

{'columns': {'cliente_id': {'sdtype': 'id'},
  'edad': {'sdtype': 'numerical'},
  'ingreso_mensual': {'sdtype': 'numerical'},
  'ciudad': {'sdtype': 'categorical'}},
 'primary_key': 'cliente_id',
 'METADATA_SPEC_VERSION': 'SINGLE_TABLE_V1'}

In [12]:
# Guardar la metadata en JSON
metadata.save_to_json(
    "dfClient_metadata.json"
)

In [13]:
# Entrenar el modelo para grenerar los datos sintéticos
synthesizer = GaussianCopulaSynthesizer(
    metadata
)

/home/karl/anaconda3/envs/ECBD/lib/python3.11/site-packages/sdv/single_table/base.py:182: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)


In [14]:
# Entrenamiento
synthesizer.fit(
    dfClient
)

In [15]:
# Generar los datos sintéticos
clientes_sinteticos = synthesizer.sample(
    num_rows = 100
)

In [16]:
clientes_sinteticos.head()

,cliente_id,edad,ingreso_mensual,ciudad
0,16169768,59,14778,Fortin
1,4918803,38,32141,Cordoba
2,1900081,36,19072,Veracruz
3,531516,30,28659,Veracruz
4,10211768,51,28681,Fortin


In [17]:
clientes_sinteticos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   cliente_id       100 non-null    int64 
 1   edad             100 non-null    int64 
 2   ingreso_mensual  100 non-null    int64 
 3   ciudad           100 non-null    object
dtypes: int64(3), object(1)
memory usage: 3.3+ KB


In [18]:
clientes_sinteticos.describe( include = "all" )

,cliente_id,edad,ingreso_mensual,ciudad
count,1.000000e+02,100.000000,100.000000,100
unique,NaN,NaN,NaN,9
top,NaN,NaN,NaN,Cordoba
freq,NaN,NaN,NaN,61
mean,8.532170e+06,42.900000,21678.150000,NaN
std,4.687773e+06,11.846237,8671.615725,NaN
min,4.660500e+04,28.000000,6185.000000,NaN
25%,5.008268e+06,33.000000,14356.750000,NaN
50%,8.656048e+06,41.000000,21358.500000,NaN
75%,1.256688e+07,52.500000,28983.000000,NaN


In [19]:
# Crear un dataframe contaminado
dfContaminado = clientes_sinteticos.copy()

In [20]:
# Colocar edades imposibles
indices = random.sample(list(dfContaminado.index), 5)

In [21]:
dfContaminado.loc[indices, "edad" ] = -5

In [22]:
# Introducir valores nulos 
duplicados = dfContaminado.sample(10, random_state = 42)

In [23]:
dfContaminado = pd.concat([dfContaminado, duplicados], ignore_index = True )

In [24]:
# Verificar valores nulos
dfContaminado.duplicated().sum()

np.int64(10)

In [25]:
dfContaminado.describe( include = "all" )

,cliente_id,edad,ingreso_mensual,ciudad
count,1.100000e+02,110.000000,110.000000,110
unique,NaN,NaN,NaN,9
top,NaN,NaN,NaN,Cordoba
freq,NaN,NaN,NaN,70
mean,8.614045e+06,38.954545,21341.318182,NaN
std,4.775996e+06,16.167351,8625.934017,NaN
min,4.660500e+04,-5.000000,6185.000000,NaN
25%,4.948624e+06,30.000000,14254.250000,NaN
50%,8.956428e+06,38.500000,20550.500000,NaN
75%,1.274153e+07,50.750000,28813.750000,NaN


In [26]:
# Generar el reporte
reporte = QualityReport()

In [27]:
reporte.generate(
    real_data = dfClient,
    synthetic_data = clientes_sinteticos,
    metadata = metadata.to_dict()
)

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 4/4 [00:00<00:00, 408.17it/s]|
Column Shapes Score: 66.33%

(2/2) Evaluating Column Pair Trends: |██████████| 6/6 [00:00<00:00, 136.93it/s]|
Column Pair Trends Score: 8.5%

Overall Score (Average): 37.42%



In [28]:
reporte.generate(
    real_data = dfClient,
    synthetic_data = dfContaminado,
    metadata = metadata.to_dict()
)

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 4/4 [00:00<00:00, 365.80it/s]|
Column Shapes Score: 64.55%

(2/2) Evaluating Column Pair Trends: |██████████| 6/6 [00:00<00:00, 137.02it/s]|
Column Pair Trends Score: 3.18%

Overall Score (Average): 33.86%

